# Figure 4 — Kingman, Real Sweep

Real-data analogue of Figure 3. Same two-panel layout (recovery curves +
$\hat p^*(n)$ scaling), but the similarity matrix is generated from JC69
sequences on Kingman trees (`interactive_run.py` sweep), and the tree-feature
parameters $(\eta, \rho)$ are **estimated from each $M$** rather than preset.

**Bound under test (non-balanced CBM):**
$$
p > \frac{C \cdot \eta(1+\eta)^3 }{(\rho - \eta\, S_{\mathrm{out}}^{\max})^2} \frac{\ln n}{n}
$$
with $\rho = (S_{\mathrm{in}}^{\max} - S_{\mathrm{out}}^{\max}) - (S_{\mathrm{out}}^{\max} - S_{\mathrm{out}}^{\min})$.

**Sweep already done.** This notebook only loads results and the cached $M$/$v_{\text{pop}}$;
no experiments are re-run.

## Cell 1 — Configuration

In [ ]:
import sys
import json
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

ROOT = Path.cwd()
while ROOT.parent != ROOT and not (ROOT / "setup.py").exists():
    ROOT = ROOT.parent
PROJECT_ROOT = ROOT / "sub_sampled_fielder_vec"
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

from analysis.theoretical_interpretation.utils import (
    imbalance_eta, n_min, structural_margin_rho, estimate_features_from_M,
    coherence_mu, compute_top_eigenpairs, spectral_gap,
    find_threshold_p_star,
)

CACHE_ROOT = PROJECT_ROOT / "src" / "cache"
SEQ_LEN    = 10_000
RECOVERY_THRESHOLD = 0.95

# Datasets to compare. Each has its own (tree_model, µ, N grid, run dir),
# so most cells loop over this list and key everything by ds["name"]
# (which must be unique). cache_dir lookup still uses tree_model + mu.
DATASETS = [
    dict(
        name="kingman_mean",
        tree_model="kingman_mean",
        mu=0.1,
        run_dir=PROJECT_ROOT / "results" / "kingman_mean" / "uniform"
            / "20260501-194101-kingman_mean_n500-8000_mu_0p1_uniform",
        n_values=[500, 1000, 2000, 4000, 6000, 8000],
        marker="o",
    ),
    dict(
        name="kingman_mu0.3",
        tree_model="kingman",
        mu=0.3,
        run_dir=PROJECT_ROOT / "results" / "kingman" / "uniform"
            / "20260501-153602-kingman_n512-4096_mu_0p3_uniform",
        n_values=[512, 1024, 2048, 4096],
        marker="s",
    ),
    dict(
        name="kingman_mu0.1",
        tree_model="kingman",
        mu=0.1,
        run_dir=PROJECT_ROOT / "results" / "kingman" / "uniform"
            / "20260505-222310-kingman_n500-8000_mu_0p1_uniform",
        # n=8000 still running — leave it out for now.
        n_values=[500, 1000, 2000, 4000, 6000],
        marker="^",
    ),
]

def cache_dir_for(tree_model: str, mu: float, n: int) -> Path:
    return CACHE_ROOT / (
        f"n{n}_L{SEQ_LEN}_mu{mu:.3f}_{tree_model}_pop_size1p000_JC69_num_classes4"
    )

def results_path_for(run_dir: Path, n: int) -> Path:
    return run_dir / f"n{n}_L{SEQ_LEN}" / "results.json"

for ds in DATASETS:
    for n in ds["n_values"]:
        rp = results_path_for(ds["run_dir"], n)
        cd = cache_dir_for(ds["tree_model"], ds["mu"], n)
        assert rp.exists(), f"missing results: {rp}"
        assert cd.exists(),  f"missing cache:   {cd}"
    print(f"{ds['name']:18s} tree={ds['tree_model']:13s} µ={ds['mu']}  n={ds['n_values']}")


## Cell 2 — Load the sweep results

`results.json` is column-oriented: a flat `columns` list and a list of `rows`,
one row per p-value. We keep `partition_agreement_M` (already in $[0, 100]$)
and convert to a fraction in $[0, 1]$ for thresholding.

In [ ]:
frames = []
for ds in DATASETS:
    for n in ds["n_values"]:
        with open(results_path_for(ds["run_dir"], n)) as f:
            payload = json.load(f)
        df_n = pd.DataFrame(payload["rows"], columns=payload["columns"])
        df_n = df_n[["p", "partition_agreement_M"]].copy()
        df_n["agreement"] = df_n["partition_agreement_M"] / 100.0
        df_n["n"] = n
        df_n["name"] = ds["name"]
        frames.append(df_n)

df_agg = (
    pd.concat(frames, ignore_index=True)
      .sort_values(["name", "n", "p"])
      .reset_index(drop=True)
)
print(f"df_agg shape: {df_agg.shape}")
df_agg.groupby(["name", "n"])["partition_agreement_M"].agg(["min", "max"])


## Cell 3 — Load cached $M$ and reference Fiedler $v_{\text{pop}}$

These are produced by the experiment runner during the original sweep and
stored under `src/cache/<key>/`. We don't recompute them — just load the
`.npz` files.

In [ ]:
matrices = {}   # ds["name"] -> {n: (M, v_pop)}
for ds in DATASETS:
    for n in ds["n_values"]:
        cdir = cache_dir_for(ds["tree_model"], ds["mu"], n)
        M     = np.load(cdir / "similarity_matrix.npz")["similarity_matrix"]
        v_pop = np.load(cdir / "fiedler_ref.npz")["fiedler_ref"]
        matrices[(ds["name"], n)] = (M, v_pop)
        print(f"{ds['name']:18s} n={n:>4}: M.shape={M.shape}, "
              f"v_pop range=[{v_pop.min():+.4f}, {v_pop.max():+.4f}]")


## Cell 4 — Estimate tree-feature parameters from each $M$

Use the sign of $v_{\text{pop}}$ to bipartition the rows/columns of $M$
into two clans, then read off in-block / cross-block similarity statistics:

- $\eta$ = `imbalance_eta(n_1, n_2)`
- $\rho$ = `structural_margin_rho(S_{\mathrm{in}}^{\max}, S_{\mathrm{out}}^{\max}, S_{\mathrm{out}}^{\min})`
- margin = $\rho - \eta \, S_{\mathrm{out}}^{\max}$  (must be $> 0$ for the bound to be finite)

In [ ]:
feature_rows = []
for ds in DATASETS:
    for n in ds["n_values"]:
        M, v_pop = matrices[(ds["name"], n)]
        feats = estimate_features_from_M(M, v_pop)
        feats["n"] = n
        feats["name"] = ds["name"]
        feats["tree_model"] = ds["tree_model"]
        feats["mu"] = ds["mu"]
        feature_rows.append(feats)

df_features = pd.DataFrame(feature_rows)[
    ["name", "tree_model", "mu", "n", "n1", "n2", "eta",
     "S_in_max", "S_out_max", "S_out_min", "rho", "margin"]
]

bad = df_features[df_features["margin"] <= 0]
if not bad.empty:
    print("⚠  Margin ≤ 0 (non-balanced CBM bound is singular) at:")
    for _, row in bad.iterrows():
        print(f"    {row['name']:18s} n={int(row['n']):>4}: "
              f"η={row['eta']:7.3f}, ρ={row['rho']:.4f}, "
              f"margin={row['margin']:.4f} (η·S_out_max={row['eta']*row['S_out_max']:.4f} > ρ)")
    print("   These rows will be excluded from the C-fit and the theory overlay.")

df_features


## Cell 5 — Top-3 eigenpairs of $L = D - M$ per $n$

For each $n$ we form the unnormalized Laplacian $L = D - M$ and extract its
three smallest eigenpairs via `compute_top_eigenpairs`. The Fiedler eigenvector
$v^{(1)}$ defines the partition; $\lambda_2$ and $\lambda_3$ feed the spectral
gap $\Delta\lambda$. Mirrors Figure 2 Cell 4.


In [ ]:
eig_per = {}
for ds in DATASETS:
    for n in ds["n_values"]:
        M, _ = matrices[(ds["name"], n)]
        L = np.diag(M.sum(axis=1)) - M
        eigvals, eigvecs = compute_top_eigenpairs(L, k=3)
        eig_per[(ds["name"], n)] = dict(
            eigvals=eigvals,
            U=eigvecs[:, :2],
            v1=eigvecs[:, 1],
        )
        print(f"{ds['name']:18s} n={n:5d}: "
              f"λ1={eigvals[0]:.4f}, λ2={eigvals[1]:.4f}, λ3={eigvals[2]:.4f}")


## Cell 6 — Linalg features $\mu(U)$, $\lambda_2$, $\Delta\lambda$ vs CBM predictions

Three relations from Thesis V.03 evaluated on real $M$:

1. $\mu(U) \approx (1+\eta)/2$ (rank-2 coherence; identity on flat CBM)
2. $\lambda_2 \approx n \cdot S_{\text{out}}^{\max}$ (Fiedler eigenvalue; identity on flat CBM)
3. Lemma 0.4: $\Delta\lambda \ge n_{\min}\,(\rho - \eta\,S_{\text{out}}^{\max})$ (lower bound; should hold strictly)

The first two collapse to identities only when $M$ is exactly flat-CBM, so on
real Kingman data they read as approximations. The third is an inequality and
is the main object of interest.


In [ ]:
mu_emp = []
lambda2_emp = []
delta_emp = []
mu_pred = []
lambda2_pred = []
gap_lb = []
for _, row in df_features.iterrows():
    n = int(row["n"])
    info = eig_per[(row["name"], n)]
    mu_emp.append(coherence_mu(info["U"]))
    lambda2_emp.append(float(info["eigvals"][1]))
    delta_emp.append(spectral_gap(info["eigvals"]))

    eta = row["eta"]
    nmin = n_min(int(row["n1"]), int(row["n2"]))
    mu_pred.append((1.0 + eta) / 2.0)
    lambda2_pred.append(n * row["S_out_max"])
    gap_lb.append(nmin * (row["rho"] - eta * row["S_out_max"]))

df_features["mu"] = mu_emp
df_features["mu_pred"] = mu_pred
df_features["lambda2"] = lambda2_emp
df_features["lambda2_pred"] = lambda2_pred
df_features["delta_lambda"] = delta_emp
df_features["gap_lb"] = gap_lb

df_features[
    ["name", "n", "eta", "mu", "mu_pred",
     "lambda2", "lambda2_pred",
     "delta_lambda", "gap_lb"]
]


## Cell 7 — Three-panel identity scatter (real-data analogue of Figure 2 Cell 12)

Each panel plots the empirical linalg feature against its CBM prediction.
Panels 1–2 should cluster near the $y=x$ identity if the real $M$ is close
to flat-CBM; panel 3 (Lemma 0.4) is a lower bound, so empirical points should
lie **above** the identity line.


In [ ]:
panels = [
    ("mu_pred", "mu", r"$(1+\eta)/2$", r"$\mu(U)$",
     r"Coherence: $\mu(U) \approx (1+\eta)/2$"),
    ("lambda2_pred", "lambda2", r"$n \cdot S_{\mathrm{out}}^{\max}$", r"$\lambda_2$",
     r"Fiedler eigenvalue: $\lambda_2 \approx n\,S_{\mathrm{out}}^{\max}$"),
    ("gap_lb", "delta_lambda",
     r"$n_{\min}(\rho - \eta\,S_{\mathrm{out}}^{\max})$", r"$\Delta\lambda$",
     r"Lemma 0.4 lower bound (points should lie above $y=x$)"),
]

# One 3-panel identity figure per dataset.
for ds in DATASETS:
    name = ds["name"]
    sub = df_features[df_features["name"] == name]
    if sub.empty:
        continue

    fig, axes = plt.subplots(1, 3, figsize=(14, 4.5), constrained_layout=True)
    fig.suptitle(fr"Cell 7 — identity scatter — {name} ($\mu={ds['mu']}$)")

    for ax, (xcol, ycol, xlbl, ylbl, title) in zip(axes, panels):
        ax.scatter(sub[xcol], sub[ycol],
                   c=sub["n"], cmap="viridis",
                   s=70, edgecolor="k", zorder=3)
        for _, r in sub.iterrows():
            ax.annotate(int(r["n"]), (r[xcol], r[ycol]),
                        textcoords="offset points", xytext=(6, 4), fontsize=8)
        x = sub[xcol].to_numpy()
        y = sub[ycol].to_numpy()
        lo = float(min(x.min(), y.min()))
        hi = float(max(x.max(), y.max()))
        pad = 0.05 * (hi - lo if hi > lo else 1.0)
        ax.plot([lo - pad, hi + pad], [lo - pad, hi + pad], "k--", lw=1, label="$y = x$")
        ax.set_xlabel(xlbl)
        ax.set_ylabel(ylbl)
        ax.set_title(title)
        ax.grid(alpha=0.3)
        ax.legend(loc="best", fontsize=9)

    plt.show()


## Cell 9 — Empirical $\hat p^*(n)$ from `partition_agreement_M`

`partition_agreement_M` is symmetric under bipartition label-flip, so it
bottoms out near 50% (random guess) at intermediate p-values rather than
monotonically rising. `find_threshold_p_star` returns the *smallest* p with
agreement $\ge 0.95$. We additionally print the curve to confirm there's no
spurious low-p crossing.

In [ ]:
p_star_per = {}
for ds in DATASETS:
    for n in ds["n_values"]:
        sub = df_agg[(df_agg.name == ds["name"]) & (df_agg.n == n)].sort_values("p")
        p_star = find_threshold_p_star(
            sub["p"].to_numpy(),
            sub["agreement"].to_numpy(),
            threshold=RECOVERY_THRESHOLD,
        )
        p_star_per[(ds["name"], n)] = p_star
        crossing = int((sub["agreement"] >= RECOVERY_THRESHOLD).sum())
        print(f"{ds['name']:18s} n={n:>4}: "
              f"p_star={p_star},  #(p with agreement ≥ 0.95) = {crossing}")

df_features["p_star"] = df_features.apply(
    lambda r: p_star_per[(r["name"], int(r["n"]))], axis=1
)
df_features


## Cell 10 — Theoretical scale and fit constant $C$

$$
\text{theory\_scale}(n) \;=\; \frac{\eta(n)\,(1+\eta(n))^3 \,\ln n}{n \cdot \text{margin}(n)^2}
$$
$$
C \;=\; \operatorname{median}_{n \,\in\, \text{valid}} \frac{\hat p^*(n)}{\text{theory\_scale}(n)}
$$
Rows with non-positive margin or `NaN` $\hat p^*$ are excluded from the median.

In [ ]:
df_features["theory_scale"] = (
    df_features["eta"] * (1 + df_features["eta"]) ** 3 * np.log(df_features["n"])
    / (df_features["n"] * df_features["margin"] ** 2)
)

C_per_dataset = {}
for name, sub in df_features.groupby("name"):
    valid = sub[(sub["margin"] > 0) & sub["p_star"].notna()]
    if valid.empty:
        C_per_dataset[name] = float("nan")
        print(f"⚠  {name}: no valid rows for C-fit — leaving C = NaN.")
    else:
        ratios = valid["p_star"] / valid["theory_scale"]
        C_per_dataset[name] = float(np.median(ratios))
        print(f"{name:18s}: C = {C_per_dataset[name]:.4g}  "
              f"(median over {len(valid)} valid n: "
              + ", ".join(str(int(x)) for x in valid['n'].tolist()) + ")")

df_features


## Cell 11 — Two-panel figure

**Left:** `partition_agreement_M` vs $p$ per $n$ (viridis), red dashed 95%
threshold.  
**Right:** $\hat p^*(n)$ vs $n$ (log-log) with the theoretical curve
$\;C\cdot\eta(n)(1+\eta(n))^3\ln n / [n\,\text{margin}(n)^2]\;$
evaluated per-$n$ (because $\eta$ and $\rho$ are estimated, not fixed).

In [ ]:
# One two-panel figure per dataset; rendered inline only (not saved).
for ds in DATASETS:
    name = ds["name"]
    sub_features = df_features[df_features["name"] == name]
    sub_agg = df_agg[df_agg["name"] == name]
    if sub_features.empty or sub_agg.empty:
        continue

    fig, (ax_l, ax_r) = plt.subplots(1, 2, figsize=(14, 5))
    fig.suptitle(
        fr"{name} real sweep — $\mu={ds['mu']}$, $L={SEQ_LEN}$, sampling=uniform, "
        fr"metric=partition_agreement_M"
    )

    # ---------- Left: recovery curves ----------
    cmap = plt.get_cmap("viridis")
    n_values = ds["n_values"]
    n_to_color = {n: cmap(i / max(1, len(n_values) - 1)) for i, n in enumerate(n_values)}

    for n in n_values:
        sub = sub_agg[sub_agg.n == n].sort_values("p")
        p_star = p_star_per[(name, n)]
        p_star_label = (f"{p_star:.4f}"
                        if p_star is not None and np.isfinite(p_star)
                        else "n/a")
        eta_n = sub_features.loc[sub_features.n == n, "eta"].iloc[0]
        ax_l.plot(
            sub["p"], sub["partition_agreement_M"],
            "-o", color=n_to_color[n], ms=4, lw=1,
            label=fr"n={n}, $\eta$={eta_n:.2f}, $\hat p^*$={p_star_label}",
        )

    ax_l.axhline(100.0 * RECOVERY_THRESHOLD, color="red", ls="--", lw=1,
                 label=f"{int(RECOVERY_THRESHOLD * 100)}% threshold")
    ax_l.set_xscale("log")
    ax_l.set_xlabel(r"Sampling probability $p$")
    ax_l.set_ylabel("partition_agreement_M (%)")
    ax_l.set_title(fr"Recovery curves by $n$ — {name}")
    ax_l.legend(fontsize=8, loc="lower right")
    ax_l.grid(True, which="both", alpha=0.3)

    # ---------- Right: p*(n) scaling ----------
    df_emp = sub_features.dropna(subset=["p_star"]).sort_values("n")
    if not df_emp.empty:
        ax_r.plot(df_emp["n"], df_emp["p_star"], "-o",
                  label=r"Empirical $\hat p^*$")

    df_thy = sub_features[sub_features["margin"] > 0].sort_values("n")
    C = C_per_dataset.get(name, float("nan"))
    if np.isfinite(C) and not df_thy.empty:
        theory = C * df_thy["theory_scale"]
        ax_r.plot(
            df_thy["n"], theory, "--",
            label=(
                r"$C\cdot\eta(1{+}\eta)^3 \ln n / [n\,\mathrm{margin}^2]$"
                f", C={C:.3g}"
            ),
        )

    ax_r.set_xscale("log")
    ax_r.set_yscale("log")
    ax_r.set_xlabel(r"$n$ (number of taxa)")
    ax_r.set_ylabel(r"$\hat p^*$")
    ax_r.set_title(fr"Scaling: $\hat p^*(n)$ vs theoretical bound — {name}")
    ax_r.legend(fontsize=9)
    ax_r.grid(True, which="both", alpha=0.3)

    fig.tight_layout(rect=(0, 0, 1, 0.95))
    plt.show()
